# openfront-rl — Kaggle burst training

Continues the `gamma=0.995` experiment on Kaggle. Resumes from the checkpoint
carried in the input Dataset and writes an updated one to `/kaggle/working`,
so sessions chain across the 12h cap.

**Read this before running:**

- This workload is **CPU-bound, not GPU-bound**. Measured locally: GPU 6–7%
  utilised, while 10 Node subprocesses simulating OpenFront games consume
  ~2.7 cores. Kaggle gives **4 vCPU vs 12 locally**, so expect this to run
  *slower* per update than the local machine, not faster. The GPU is close to
  irrelevant here.
- `NUM_ENVS` is therefore set to **3**, not the local 10. Ten Node workers
  contending over 4 vCPU makes every `VecEnv.step()` wait on the slowest one.
- Fewer envs means less diverse rollouts per update, so results are **not**
  directly comparable to the local run at equal update count.

**Required:** attach the `openfront-rl-src` Dataset (built by
`kaggle/package_source.sh`) as input, and enable Internet (the notebook
clones OpenFrontIO and runs `npm ci`).

In [ ]:
import os, subprocess, sys, time, shutil, pathlib

WORK = "/kaggle/working/openfront-rl"
OUT  = "/kaggle/working/out"            # what the next session picks up

# Stop training with margin before Kaggle's hard 12h kill. train.py
# checkpoints every --checkpoint-every updates, so a clean stop loses at
# most that many updates; a hard kill at the cap could land mid-write.
SESSION_BUDGET_H = 11.0
NUM_ENVS = 3          # 4 vCPU -- see the note above
GAMMA = 0.995
MAX_EPISODE_STEPS = 2500

print("CPUs:", os.cpu_count())
print(subprocess.run(["free","-g"], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no GPU")

# Locate the input Dataset by CONTENT (searching for the PINNED_SHA marker
# file) rather than assuming a fixed path depth. This has already broken
# once on a bad assumption: the first version asserted a hardcoded
# /kaggle/input/<slug> path and failed even though the Dataset was attached,
# because the mount was not ready yet in that session. The second version
# listed /kaggle/input directly and found only a "datasets" entry one level
# above where the files actually are -- Kaggle nests kernel-attached inputs
# as /kaggle/input/datasets/<owner>/<slug>/... on this account/kernel type,
# not the flat /kaggle/input/<slug>/ most docs show. Walking the tree
# (bounded depth, so a bad mount fails fast rather than hanging) is immune
# to either layout.
INPUT_ROOT = "/kaggle/input"


def find_dataset_root(root: str, max_depth: int = 4) -> str | None:
    if not os.path.isdir(root):
        return None
    start_depth = root.rstrip("/").count("/")
    for dirpath, dirnames, filenames in os.walk(root):
        if "PINNED_SHA" in filenames:
            return dirpath
        if dirpath.rstrip("/").count("/") - start_depth >= max_depth:
            dirnames[:] = []  # do not descend further from here
    return None


def list_tree(root: str, max_depth: int = 3) -> list[str]:
    out = []
    if not os.path.isdir(root):
        return out
    start_depth = root.rstrip("/").count("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.rstrip("/").count("/") - start_depth
        if depth <= max_depth:
            out.append(dirpath)
        if depth >= max_depth:
            dirnames[:] = []
    return out


print("\ninput tree (depth <=3):")
for p in list_tree(INPUT_ROOT):
    print(" ", p)

DATASET = find_dataset_root(INPUT_ROOT)
if DATASET is None:
    raise SystemExit(
        "No input Dataset containing PINNED_SHA found under /kaggle/input "
        "(searched 4 levels deep).\n"
        "Fix: in the notebook sidebar, Add Input -> owenkarmel/openfront-rl-src, "
        "then re-run. If it was just uploaded, wait for it to finish processing "
        "first -- a Dataset attached before it is ready will not mount."
    )

print("\nusing dataset:", DATASET)
print("contents:", sorted(os.listdir(DATASET)))


## Setup

Clones OpenFrontIO at the pinned SHA, lays out the directory tree the code
expects, installs Node deps, and runs the smoke test. The smoke test is the
point of no return — if it fails, nothing below will work.

In [ ]:
os.makedirs(OUT, exist_ok=True)
setup = os.path.join(DATASET, "setup_kaggle.sh")
assert os.path.exists(setup), f"setup_kaggle.sh missing from dataset: {sorted(os.listdir(DATASET))}"
subprocess.run(["bash", setup, DATASET, WORK], check=True)


## Train

Runs `train.py` as a subprocess and stops it at the session budget. Killing
it is safe by design: the run resumes from `latest.pt`, so a stop costs at
most `--checkpoint-every` updates.

In [ ]:
import signal

log_path = os.path.join(WORK, "training/checkpoints/train_stdout.log")
cmd = [
    sys.executable, "-u", "train.py",
    "--num-envs", str(NUM_ENVS),
    "--rollout-length", "300",
    "--gamma", str(GAMMA),
    "--max-episode-steps", str(MAX_EPISODE_STEPS),
    "--entropy-coef", "0.02",
    "--minibatch-size", "128",
    "--checkpoint-every", "10",
    "--eval-every", "10",
]
print(" ".join(cmd))

logf = open(log_path, "a")
proc = subprocess.Popen(cmd, cwd=os.path.join(WORK, "training"),
                        stdout=logf, stderr=subprocess.STDOUT)

deadline = time.time() + SESSION_BUDGET_H * 3600
last_size = 0
try:
    while proc.poll() is None and time.time() < deadline:
        time.sleep(300)
        # Stream progress into the notebook output so the session is not a
        # black box for 11 hours.
        with open(log_path) as fh:
            fh.seek(last_size)
            new = fh.read()
            last_size = fh.tell()
        tail = [l for l in new.splitlines() if l.startswith("update")][-2:]
        for l in tail:
            print(l, flush=True)
finally:
    if proc.poll() is None:
        print("session budget reached -- stopping training cleanly", flush=True)
        proc.send_signal(signal.SIGTERM)
        try:
            proc.wait(timeout=120)
        except subprocess.TimeoutExpired:
            proc.kill()
    logf.close()
print("exit code:", proc.returncode)

## Save state for the next session

Copies the checkpoint and the accumulated logs into `/kaggle/working/out`.
Push that back as a new version of the input Dataset so the next session
resumes where this one stopped:

```bash
kaggle datasets version -p out/ -m "session N"
```

Replays are **not** copied by default — they accumulate to hundreds of MB and
are only needed for replay analysis, not for resuming.

In [ ]:
ck = os.path.join(WORK, "training/checkpoints")
os.makedirs(os.path.join(OUT, "checkpoint"), exist_ok=True)
for f in ["latest.pt", "notified_milestones.json", "train_log.csv", "train_stdout.log"]:
    src = os.path.join(ck, f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUT, "checkpoint", f))
        print(f"saved {f}: {os.path.getsize(src)/1e6:.1f} MB")

# Carry the source + pin forward so the output Dataset is self-contained and
# can be attached directly to the next session.
for d in ["env-bridge", "training"]:
    shutil.copytree(os.path.join(DATASET, d), os.path.join(OUT, d), dirs_exist_ok=True)
shutil.copy2(os.path.join(DATASET, "PINNED_SHA"), os.path.join(OUT, "PINNED_SHA"))
for f in ["dataset-metadata.json"]:
    p = os.path.join(DATASET, f)
    if os.path.exists(p):
        shutil.copy2(p, os.path.join(OUT, f))

print("\nlast updates this session:")
print(subprocess.run(["tail", "-5", os.path.join(ck, "train_stdout.log")],
                     capture_output=True, text=True).stdout)